In [3]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [36]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Historical Share Price", "Historical Revenue"), vertical_spacing = .3)
    stock_data_specific = stock_data[stock_data.Date <= '2023--03-27']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2022-12-31']
    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data_specific.Date, infer_datetime_format=True), y=stock_data_specific.Close.astype("float"), name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data_specific.Date, infer_datetime_format=True), y=revenue_data_specific.Revenue.astype("float"), name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False,
    height=900,
    title=stock,
    xaxis_rangeslider_visible=True)
    fig.show()

In [4]:
import plotly.io as pio
pio.renderers.default = "iframe"

In [5]:
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [6]:
tesla = yf.Ticker("TSLA")  # Using the ticker symbol "TSLA" for Tesla

In [7]:
tesla_data = tesla.history(period="max")  # We get the information for the maximun amount of time

In [8]:
tesla_data.reset_index(inplace=True)   # Resetting the index
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


In [39]:
tesla_data.tail()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
3851,2025-10-20 00:00:00-04:00,443.869995,449.799988,440.609985,447.429993,63719000,0.0,0.0
3852,2025-10-21 00:00:00-04:00,445.760010,449.299988,442.049988,442.600006,54412200,0.0,0.0
3853,2025-10-22 00:00:00-04:00,443.450012,445.540009,429.000000,438.970001,84023500,0.0,0.0
3854,2025-10-23 00:00:00-04:00,420.000000,449.399994,413.899994,448.980011,126709800,0.0,0.0
3855,2025-10-24 00:00:00-04:00,446.829987,451.679993,430.170013,433.720001,94408400,0.0,0.0


In [9]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data  = requests.get(url).text

In [10]:
soup = BeautifulSoup(html_data, 'html5lib')   # parse the html_data using BeautifulSoup 

In [12]:
tables = soup.find_all('table')   #find all html tables in the web page
len(tables)      # we can see how many tables were found by checking the length of the tables list

6

In [13]:
for index,table in enumerate(tables):
    if ("Tesla Quarterly Revenue" in str(table)):         # to get the index of table with Tesla Quarterly Revenue
        table_index = index
print(table_index)

1


In [14]:
tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in tables[table_index].tbody.find_all("tr"):
    col = row.find_all("td")
    if (col != []):
        date = col[0].text
        revenue = col[1].text
        data = pd.DataFrame.from_dict({"Date": [date], "Revenue": [revenue]})
        tesla_revenue = pd.concat([tesla_revenue, data], ignore_index = True)
                                      
tesla_revenue

,Date,Revenue
0,2022-09-30,"$21,454"
1,2022-06-30,"$16,934"
2,2022-03-31,"$18,756"
3,2021-12-31,"$17,719"
4,2021-09-30,"$13,757"
5,2021-06-30,"$11,958"
6,2021-03-31,"$10,389"
7,2020-12-31,"$10,744"
8,2020-09-30,"$8,771"
9,2020-06-30,"$6,036"


In [15]:
dataframe_list = pd.read_html(url, flavor='bs4')  # passing the url data as a dataframe using read_html

In [16]:
tesla_revenue = dataframe_list[1]  # storing our data in a variable using index of 1
tesla_revenue.columns = ["Date", "Revenue"]  # Naming the columns Date and Revenue
tesla_revenue

,Date,Revenue
0,2022-09-30,"$21,454"
1,2022-06-30,"$16,934"
2,2022-03-31,"$18,756"
3,2021-12-31,"$17,719"
4,2021-09-30,"$13,757"
5,2021-06-30,"$11,958"
6,2021-03-31,"$10,389"
7,2020-12-31,"$10,744"
8,2020-09-30,"$8,771"
9,2020-06-30,"$6,036"


In [20]:
tesla_revenue["Revenue"] = tesla_revenue['Revenue'].str.replace(r',|\$',"",regex=True)  # Removing the comma and dollar sign from Revenue column
tesla_revenue.dropna(inplace=True)  # Removing null
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]  # Removing empty strings

In [21]:
tesla_revenue.head()

,Date,Revenue
0,2022-09-30,21454
1,2022-06-30,16934
2,2022-03-31,18756
3,2021-12-31,17719
4,2021-09-30,13757


In [40]:
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


In [22]:
gamestop = yf.Ticker("GME")

In [23]:
gme_data = gamestop.history(period="max")

In [24]:
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658001,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615921,1.662210,1.603296,1.662210,6892800,0.0,0.0


In [25]:
url ='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html'
html_data  = requests.get(url).text

In [26]:
gme_soup = BeautifulSoup(html_data, 'html5lib')

In [27]:
tables = gme_soup.find_all('table') 
len(tables)

6

In [28]:
for index,table in enumerate(tables):
    if ("Gamestop Quarterly Revenue" in str(table)):         # to get the index of table with Gamestop Quarterly Revenue
        table_index = index
print(table_index)

1


In [29]:
gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for row in tables[table_index].tbody.find_all("tr"):
    col = row.find_all("td")
    if (col != []):
        date = col[0].text
        revenue = col[1].text
        data = pd.DataFrame.from_dict({"Date": [date], "Revenue": [revenue]})
        gme_revenue = pd.concat([tesla_revenue, data], ignore_index = True)
                                      
gme_revenue

,Date,Revenue
0,2022-09-30,21454
1,2022-06-30,16934
2,2022-03-31,18756
3,2021-12-31,17719
4,2021-09-30,13757
5,2021-06-30,11958
6,2021-03-31,10389
7,2020-12-31,10744
8,2020-09-30,8771
9,2020-06-30,6036


In [30]:
dataframe_list = pd.read_html(url, flavor='bs4')
gme_revenue = dataframe_list[1]

In [31]:
gme_revenue.columns = ["Date", "Revenue"]
gme_revenue

,Date,Revenue
0,2020-04-30,"$1,021"
1,2020-01-31,"$2,194"
2,2019-10-31,"$1,439"
3,2019-07-31,"$1,286"
4,2019-04-30,"$1,548"
...,...,...
57,2006-01-31,"$1,667"
58,2005-10-31,$534
59,2005-07-31,$416
60,2005-04-30,$475


In [34]:
gme_revenue["Revenue"] = gme_revenue['Revenue'].str.replace(r',|\$',"",regex=True)
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != ""]
gme_revenue.head()

,Date,Revenue
0,2020-04-30,1021
1,2020-01-31,2194
2,2019-10-31,1439
3,2019-07-31,1286
4,2019-04-30,1548


In [41]:
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


In [37]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

C:\Users\Latitude7280\AppData\Local\Temp\ipykernel_18580\1991719827.py:5: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.

C:\Users\Latitude7280\AppData\Local\Temp\ipykernel_18580\1991719827.py:6: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.



In [38]:
make_graph(gme_data, gme_revenue, 'GameStop')

C:\Users\Latitude7280\AppData\Local\Temp\ipykernel_18580\1991719827.py:5: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.

C:\Users\Latitude7280\AppData\Local\Temp\ipykernel_18580\1991719827.py:6: UserWarning:

The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.

